# TANGO Aggregation Analysis — Aβ42 Variants

Mutations are classified relative to **Wildtype** (score = 1526.48) into 5 groups based on the change in aggregation (Δ):

| Group | Δ score |
|--------|---------|
| Strongly destabilizing | Δ < −50 |
| Weakly destabilizing | −50 ≤ Δ < −5 |
| Neutral | −5 ≤ Δ ≤ +5 |
| Enhancing | +5 < Δ ≤ +30 |
| Strongly enhancing | Δ > +30 |

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re

# ── Palette ──────────────────────────────────────────────────
BG = "#0a0c14"
SURFACE = "#111520"
MUTED = "#64748b"
TEXT = "#e2e8f0"

# Colors for the 5 groups
C_STRONG_DOWN = "#3b82f6"  # bright blue — strongly destabilizing
C_WEAK_DOWN = "#93c5fd"  # light blue   — weakly destabilizing
C_NEUTRAL = "#64748b"  # gray         — neutral
C_UP = "#fb923c"  # orange       — enhancing
C_STRONG_UP = "#ef4444"  # red          — strongly enhancing
C_WT = "#4ff7c0"  # green        — wildtype

print("Libraries loaded")

In [ ]:
import pandas as pd

# ── Read TANGO results from file ─────────────────────────
df = pd.read_csv("../predictions/tango/tango_input_aggregation.txt", sep="\t")

WT_SCORE = df.loc[df["Sequence"] == "Wildtype_Abeta_42", "Aggregation"].iloc[0]
raw = dict(zip(df["Sequence"], df["Aggregation"]))

# ── Classification thresholds ─────────────────────────────────────
THRESH = {
    "strong_down": -50,
    "weak_down": -5,
    "neutral_hi": +5,
    "up": +30,
}


def classify(delta):
    if delta < THRESH["strong_down"]:
        return "strong_down"
    elif delta < THRESH["weak_down"]:
        return "weak_down"
    elif delta <= THRESH["neutral_hi"]:
        return "neutral"
    elif delta <= THRESH["up"]:
        return "up"
    else:
        return "strong_up"


GROUP_META = {
    "strong_down": {
        "label": "Strongly destabilizing\n(Δ < −50)",
        "color": C_STRONG_DOWN,
        "order": 0,
    },
    "weak_down": {
        "label": "Weakly destabilizing\n(−50 ≤ Δ < −5)",
        "color": C_WEAK_DOWN,
        "order": 1,
    },
    "neutral": {"label": "Neutral\n(−5 ≤ Δ ≤ +5)", "color": C_NEUTRAL, "order": 2},
    "up": {"label": "Enhancing\n(+5 < Δ ≤ +30)", "color": C_UP, "order": 3},
    "strong_up": {
        "label": "Strongly enhancing\n(Δ > +30)",
        "color": C_STRONG_UP,
        "order": 4,
    },
}


def short_name(n):
    return re.sub(r"_Abeta_?42$", "", n)


# Compute Δ and classify
variants = []
for name, score in raw.items():
    if name == "Wildtype_Abeta_42":
        continue
    delta = score - WT_SCORE
    group = classify(delta)
    variants.append(
        {
            "name": name,
            "short": short_name(name),
            "score": score,
            "delta": delta,
            "group": group,
        }
    )

# Statistics
from collections import Counter

counts = Counter(v["group"] for v in variants)
print(f"Wildtype score: {WT_SCORE}")
print(f"Total mutations: {len(variants)}\n")
for gk, gm in GROUP_META.items():
    label_short = gm["label"].split("\n")[0]
    print(f"  {label_short:30s}: {counts[gk]:3d}")

In [ ]:
# Plot 1 — Horizontal lollipop chart (Δ from WT)

# Sort: first by group, then by delta within group
sorted_v = sorted(variants, key=lambda x: (GROUP_META[x["group"]]["order"], x["delta"]))

n = len(sorted_v)
fig, ax = plt.subplots(figsize=(14, n * 0.28 + 2), facecolor=BG)
ax.set_facecolor(SURFACE)

ys = list(range(n))
deltas = [v["delta"] for v in sorted_v]
colors = [GROUP_META[v["group"]]["color"] for v in sorted_v]
names = [v["short"] for v in sorted_v]

# Alternating row bands
for i in range(n):
    if i % 2 == 0:
        ax.axhspan(i - 0.5, i + 0.5, color="white", alpha=0.02, zorder=0)

# Vertical line at WT = 0
ax.axvline(0, color=C_WT, linewidth=1.2, alpha=0.6, linestyle="--", zorder=1)
ax.text(
    0,
    n + 0.3,
    "WT",
    ha="center",
    va="bottom",
    color=C_WT,
    fontsize=8,
    fontfamily="monospace",
    fontweight="bold",
)

# Threshold zones (light shading)
ax.axvspan(-520, THRESH["strong_down"], color=C_STRONG_DOWN, alpha=0.06, zorder=0)
ax.axvspan(
    THRESH["strong_down"], THRESH["weak_down"], color=C_WEAK_DOWN, alpha=0.05, zorder=0
)
ax.axvspan(
    THRESH["weak_down"], THRESH["neutral_hi"], color=C_NEUTRAL, alpha=0.04, zorder=0
)
ax.axvspan(THRESH["neutral_hi"], THRESH["up"], color=C_UP, alpha=0.05, zorder=0)
ax.axvspan(THRESH["up"], 100, color=C_STRONG_UP, alpha=0.06, zorder=0)

# Lines and points (lollipop)
for i, (y, v) in enumerate(zip(ys, sorted_v)):
    c = colors[i]
    d = v["delta"]
    ax.plot([0, d], [y, y], color=c, linewidth=0.8, alpha=0.5, zorder=2)
    ax.scatter(d, y, color=c, s=38, zorder=3, linewidths=0)

# Labels on the left
ax.set_yticks(ys)
ax.set_yticklabels(names, fontsize=6.5, fontfamily="monospace", color=TEXT)
ax.tick_params(axis="y", length=0, pad=4)

# Color the labels by group
for tick, v in zip(ax.get_yticklabels(), sorted_v):
    tick.set_color(GROUP_META[v["group"]]["color"])
    tick.set_alpha(0.85)

# X-axis
ax.set_xlabel(
    "Δ Aggregation score  (vs. Wildtype)",
    color=MUTED,
    fontsize=9,
    fontfamily="monospace",
    labelpad=8,
)
ax.tick_params(axis="x", colors=MUTED, labelsize=7)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(axis="x", bottom=True, color=MUTED)
ax.xaxis.label.set_color(MUTED)
ax.set_xlim(-560, 110)
ax.set_ylim(-1, n)

# Vertical threshold lines
for thresh_val, label_str in [
    (THRESH["strong_down"], "−50"),
    (THRESH["weak_down"], "−5"),
    (THRESH["neutral_hi"], "+5"),
    (THRESH["up"], "+30"),
]:
    ax.axvline(thresh_val, color="white", linewidth=0.4, alpha=0.15, zorder=1)
    ax.text(
        thresh_val,
        -1.4,
        label_str,
        ha="center",
        va="top",
        color=MUTED,
        fontsize=6,
        fontfamily="monospace",
    )

# Legend
legend_patches = [
    mpatches.Patch(facecolor=C_WT, label=f"Wildtype  ({WT_SCORE})"),
    mpatches.Patch(
        facecolor=C_STRONG_DOWN,
        label=f"Strongly destabilizing  (Δ < −50)  · n={counts['strong_down']}",
    ),
    mpatches.Patch(
        facecolor=C_WEAK_DOWN,
        label=f"Weakly destabilizing  (−50 ≤ Δ < −5)  · n={counts['weak_down']}",
    ),
    mpatches.Patch(
        facecolor=C_NEUTRAL, label=f"Neutral  (−5 ≤ Δ ≤ +5)  · n={counts['neutral']}"
    ),
    mpatches.Patch(
        facecolor=C_UP, label=f"Enhancing  (+5 < Δ ≤ +30)  · n={counts['up']}"
    ),
    mpatches.Patch(
        facecolor=C_STRONG_UP,
        label=f"Strongly enhancing  (Δ > +30)  · n={counts['strong_up']}",
    ),
]
leg = ax.legend(
    handles=legend_patches,
    loc="lower right",
    frameon=True,
    framealpha=0.15,
    edgecolor=MUTED,
    fontsize=7.5,
    labelcolor=TEXT,
    facecolor=SURFACE,
    bbox_to_anchor=(1.0, 0.0),
)

ax.set_title(
    "TANGO Aggregation  ·  Δ vs. Wildtype Aβ42",
    color="white",
    fontsize=12,
    fontfamily="monospace",
    fontweight="bold",
    pad=14,
)

plt.tight_layout()
plt.savefig(
    "tango_lollipop.png", dpi=180, bbox_inches="tight", facecolor=BG, edgecolor="none"
)
print("Saved: tango_lollipop.png")
plt.show()

In [ ]:
# Plot 2 — Summary table by group

fig, axes = plt.subplots(
    1, 5, figsize=(22, 8), facecolor=BG, gridspec_kw={"wspace": 0.35}
)

group_order = ["strong_down", "weak_down", "neutral", "up", "strong_up"]

for ax, gk in zip(axes, group_order):
    gm = GROUP_META[gk]
    c = gm["color"]
    members = sorted(
        [v for v in variants if v["group"] == gk], key=lambda x: x["delta"]
    )

    ax.set_facecolor(SURFACE)
    for spine in ax.spines.values():
        spine.set_edgecolor(c)
        spine.set_linewidth(1.2)
        spine.set_alpha(0.4)

    # Group title
    ax.set_title(
        gm["label"],
        color=c,
        fontsize=9,
        fontfamily="monospace",
        fontweight="bold",
        pad=10,
        linespacing=1.5,
    )

    # Mutation list
    n_m = len(members)
    for j, v in enumerate(members):
        y = 1.0 - (j + 0.5) / max(n_m, 1)
        delta_str = f"{v['delta']:+.1f}"
        # name
        ax.text(
            0.04,
            y,
            v["short"],
            ha="left",
            va="center",
            color=TEXT,
            fontsize=7.5,
            fontfamily="monospace",
            transform=ax.transAxes,
            alpha=0.9,
        )
        # delta
        ax.text(
            0.96,
            y,
            delta_str,
            ha="right",
            va="center",
            color=c,
            fontsize=7.5,
            fontfamily="monospace",
            fontweight="bold",
            transform=ax.transAxes,
        )
        # separator
        if j < n_m - 1:
            ax.axhline(1.0 - (j + 1) / n_m, color="white", alpha=0.04, linewidth=0.5)

    # Counter at bottom
    ax.text(
        0.5,
        -0.06,
        f"n = {n_m}",
        ha="center",
        va="top",
        color=c,
        fontsize=9,
        fontfamily="monospace",
        fontweight="bold",
        transform=ax.transAxes,
    )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xticks([])
    ax.set_yticks([])

fig.text(
    0.5,
    1.01,
    "TANGO Aggregation  ·  Classification of Aβ42 mutations",
    ha="center",
    color="white",
    fontsize=13,
    fontfamily="monospace",
    fontweight="bold",
)
fig.text(
    0.5,
    0.975,
    f"Wildtype score = {WT_SCORE}  ·  mutation name / Δ score",
    ha="center",
    color=MUTED,
    fontsize=8,
    fontfamily="monospace",
)

plt.savefig(
    "tango_groups.png", dpi=180, bbox_inches="tight", facecolor=BG, edgecolor="none"
)
print("Saved: tango_groups.png")
plt.show()